In [0]:
from pyspark.sql.functions import col, expr, current_timestamp
from delta.tables import DeltaTable

In [0]:
df_bronze_em  = spark.table("bronze.epa_emissions_raw")

In [0]:
df_silver_em = (
    df_bronze_em
      # Cast 'id' to BIGINT and create 'vehicle_id'
      .withColumn("vehicle_id", expr("try_cast(id as BIGINT)"))
      # Filter out rows where 'vehicle_id' is null
      .filter(col("vehicle_id").isNotNull())
      # Select and cast relevant columns, rename as needed
      .select(
          col("vehicle_id"),
          col("efid"),
          col("salesarea").cast("int").alias("sales_area"),
          col("score").cast("double").alias("score"),
          col("scorealt").cast("double").alias("score_alt"),
          col("smartwayscore").cast("double").alias("smartway_score"),
          col("standard"),
          col("stdtext").alias("standard_text"),
          current_timestamp().alias("ingestion_ts")
      )
      # Remove duplicates based on 'vehicle_id', 'sales_area', and 'standard'
      .dropDuplicates(["vehicle_id", "sales_area", "standard"])
)


In [0]:
# Upsert df_silver_em into the Delta table 'silver.epa_emissions'
target = "silver.epa_emissions"

if spark.catalog.tableExists(target):
    # If the target table exists, perform a merge (upsert) based on vehicle_id, sales_area, and standard
    dt = DeltaTable.forName(spark, target)
    (dt.alias("t")
      .merge(
          df_silver_em.alias("s"),
          "t.vehicle_id = s.vehicle_id AND t.sales_area = s.sales_area AND t.standard = s.standard"
      )
      .whenMatchedUpdateAll()  # Update all columns if a match is found
      .whenNotMatchedInsertAll()  # Insert all columns if no match is found
      .execute()
    )
else:
    # If the target table does not exist, create it with the data from df_silver_em
    df_silver_em.write.format("delta").mode("overwrite").saveAsTable(target)